In [ ]:
import pandas as pd

In [ ]:
features = pd.read_parquet('data/diabetes_features.parquet')
target = pd.read_parquet('data/diabetes_target.parquet')

# Columns and Data Types

In [ ]:
print(features.info())

In [ ]:
for c in features.columns.values:
    print(c)

In [ ]:
features.patient_nbr.value_counts()

In [ ]:
# Exploring a single patient
features[features.patient_nbr == 88785891]

# Missing Data

#### Key Findings

- Weight is missing for an entire patient, except for 36 patients captured below
- Missing values are encoded in some columns using "?" and None

In [ ]:
# Diagnosing if weight is always missing for a patient
weight_diagnostic = features.loc[:, ['patient_nbr', 'weight']]

weight_diagnostic['missing_weight'] = weight_diagnostic['weight'].apply(lambda x: 1 if x == "?" else 0)

# Group and aggregate
grouped = weight_diagnostic.groupby('patient_nbr').agg({
    'missing_weight': 'sum',  # count of missing weights
    'weight': 'count'          # total records for that patient
})

# Patients where missing_weight == total records are always missing
grouped['always_missing'] = grouped['missing_weight'] == grouped['weight']

grouped[(grouped["missing_weight"] != grouped['weight']) & (grouped["missing_weight"] > 0)]

In [ ]:
# Inspecting one patient that has mixed missing data
features[features.patient_nbr == 4790016]

### Diving into the use of Identifiers ("?", None) for missing data

In [ ]:
qmark_missing = (features == "?").sum()
print(qmark_missing[qmark_missing > 0])

In [ ]:
none_missing = (features == "None").sum()
print(none_missing[none_missing > 0])

#### Correlation of Missing Values

In [ ]:
((features == "?") | (features == "None")).astype(int).corr().round(4)

#### Exploring Patterns in Missing Data

Missing data in the following columns depends on admission type and admission source:

- max_glu_serum
- A1Cresult
- weight
- payer_code
- medical_specialty

This pattern suggests data is MAR with dependency on the admission source and type. A Preliminary approach is to fit the model after subsetting by admission type and source, or picking a model (i.e. tree-based) that will automatically subset the data

In [ ]:
# Admission Type
for f in ['max_glu_serum', 'A1Cresult', 'weight', 'payer_code', 'medical_specialty']:
    ct = pd.crosstab(features['admission_type_id'], features[f])
    print(ct.apply(lambda x: x / sum(x), axis = 1))


In [ ]:
# Admission Source
for f in ['max_glu_serum', 'A1Cresult', 'weight', 'payer_code', 'medical_specialty']:
    ct = pd.crosstab(features['admission_source_id'], features[f])
    print(ct.apply(lambda x: x / sum(x), axis = 1))

# Outliers